In [11]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,LabelEncoder,OneHotEncoder
from sklearn.pipeline import Pipeline
from scikeras.wrappers import KerasClassifier
from sklearn.model_selection import GridSearchCV
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping
import pickle

In [12]:
data = pd.read_csv('Churn_Modelling.csv')
data = data.drop(columns=['RowNumber','CustomerId','Surname'])

label_encoder_gender = LabelEncoder()
data['Gender'] = label_encoder_gender.fit_transform(data['Gender'])

one_hot_encoder_geo = OneHotEncoder(handle_unknown='ignore')
geo_encoded = one_hot_encoder_geo.fit_transform(data[['Geography']]).toarray()
geo_encoded_df = pd.DataFrame(geo_encoded, columns=one_hot_encoder_geo.get_feature_names_out(['Geography']))

data = pd.concat([data.drop('Geography',axis=1), geo_encoded_df], axis=1)

x = data.drop(columns=['Exited'])
y = data['Exited']

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

# Save encoder and scaler for future use

with open('label_encoder_gender.pkl', 'wb') as le_file:
    pickle.dump(label_encoder_gender, le_file)

with open('one_hot_encoder_geo.pkl', 'wb') as ohe_file:
    pickle.dump(one_hot_encoder_geo, ohe_file)

with open('scaler.pkl', 'wb') as scaler_file:
    pickle.dump(scaler, scaler_file)


In [13]:
# Define a function to create the model and try different parametere (KerasClassifier)

def create_model(optimizer='adam', activation='relu', neurons=32, layers=1):
    model = Sequential()
    model.add(Dense(neurons, input_dim=x_train.shape[1], activation=activation))
    for _ in range(layers - 1):
        model.add(Dense(neurons, activation=activation))
    model.add(Dense(1, activation='sigmoid'))
    model.compile(loss='binary_crossentropy', optimizer=optimizer, metrics=['accuracy'])
    return model




In [14]:
# Create a KerasClassifier

model = KerasClassifier(
    model=create_model,
    optimizer="adam",
    activation="relu",
    neurons=32,
    layers=1,
    epochs=50,
    batch_size=10,
    verbose=0
)


In [15]:

# Define the grid search parameters
param_grid = {
    'neurons': [16, 32, 64, 128],
    'layers': [1, 2],
    'epochs': [50, 100]
}


In [16]:
# Perform the grid search



grid = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    n_jobs=-1,
    cv=3
)

grid_result = grid.fit(x_train, y_train)

# Print the best parameters and score
print(f"Best: {grid_result.best_score_} using {grid_result.best_params_}")

AttributeError: 'super' object has no attribute '__sklearn_tags__'